# 03 — EDA & Business Analysis

Goal: answer portfolio-relevant business questions from the cleaned source-of-truth dataset. All comparisons are descriptive.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = PROJECT_ROOT / "data" / "processed" / "boston_airbnb_cleaned.csv"
df = pd.read_csv(DATA_PATH)
df.shape

(3585, 43)

## Core KPIs

In [2]:
kpis = pd.Series({
    "total_listings": len(df),
    "avg_price_usd": df["price_usd"].mean(),
    "median_price_usd": df["price_usd"].median(),
    "avg_rating": df["review_scores_rating"].mean(),
    "rated_listings": df["review_scores_rating"].notna().sum(),
    "avg_availability_365": df["availability_365"].mean(),
    "total_reviews": df["number_of_reviews"].sum(),
})
kpis

total_listings          3,585.00
avg_price_usd             173.93
median_price_usd          150.00
avg_rating                 91.92
rated_listings          2,772.00
avg_availability_365      179.35
total_reviews          68,275.00
dtype: float64

## 1. What is the listing mix by room type?

In [3]:
room_type_summary = df.groupby("room_type").agg(
    listings=("id", "count"),
    avg_price_usd=("price_usd", "mean"),
    median_price_usd=("price_usd", "median"),
    avg_rating=("review_scores_rating", "mean"),
    avg_availability_365=("availability_365", "mean")
).sort_values("listings", ascending=False)
room_type_summary["share_pct"] = room_type_summary["listings"] / len(df) * 100
room_type_summary

,listings,avg_price_usd,median_price_usd,avg_rating,avg_availability_365,share_pct
room_type,,,,,,
Entire home/apt,2127,227.44,199.00,92.33,176.26,59.33
Private room,1378,96.04,80.00,91.34,182.39,38.44
Shared room,80,92.76,60.00,90.83,208.95,2.23


## 2. Which neighbourhoods contain the most listings?

In [4]:
neighbourhood_summary = df.groupby("neighbourhood_cleansed").agg(
    listings=("id", "count"),
    avg_price_usd=("price_usd", "mean"),
    median_price_usd=("price_usd", "median"),
    avg_rating=("review_scores_rating", "mean")
).sort_values("listings", ascending=False)
neighbourhood_summary.head(10)

,listings,avg_price_usd,median_price_usd,avg_rating
neighbourhood_cleansed,,,,
Jamaica Plain,343,138.48,100.00,94.16
South End,326,204.35,180.00,93.26
Back Bay,302,240.95,209.00,91.53
Fenway,290,220.39,191.00,89.42
Dorchester,269,91.64,72.00,89.28
Allston,260,112.31,85.00,90.16
Beacon Hill,194,224.44,195.00,93.38
Brighton,185,118.77,90.00,90.86
South Boston,174,187.61,150.00,93.98


## 3. Which sufficiently represented neighbourhoods have the highest median prices?

A 20-listing threshold is used only for this ranking to avoid interpreting tiny groups as stable neighbourhood-level comparisons. No rows are deleted from the dataset.

In [5]:
neighbourhood_summary.query("listings >= 20").sort_values(["median_price_usd", "listings"], ascending=[False, False]).head(10)

,listings,avg_price_usd,median_price_usd,avg_rating
neighbourhood_cleansed,,,,
South Boston Waterfront,83,306.06,245.00,94.43
Downtown,172,236.46,225.00,92.43
Chinatown,71,232.35,219.00,92.12
Back Bay,302,240.95,209.00,91.53
Bay Village,24,266.83,206.50,93.00
West End,49,209.59,199.00,90.33
Beacon Hill,194,224.44,195.00,93.38
Fenway,290,220.39,191.00,89.42
South End,326,204.35,180.00,93.26


## 4. How does property type differ in volume and price?

In [6]:
df.groupby("property_type", dropna=False).agg(
    listings=("id", "count"),
    avg_price_usd=("price_usd", "mean"),
    median_price_usd=("price_usd", "median")
).sort_values("listings", ascending=False)

,listings,avg_price_usd,median_price_usd
property_type,,,
Apartment,2612,181.97,159.00
House,562,131.60,84.50
Condominium,231,190.25,160.00
Townhouse,54,149.04,99.00
Bed & Breakfast,41,129.37,100.00
Loft,39,185.41,166.00
Other,17,229.94,218.00
Boat,12,268.83,262.50
Villa,6,189.50,170.00


## 5. How does guest capacity relate descriptively to price?

In [7]:
df.groupby("accommodates").agg(
    listings=("id", "count"),
    avg_price_usd=("price_usd", "mean"),
    median_price_usd=("price_usd", "median")
).sort_index()

,listings,avg_price_usd,median_price_usd
accommodates,,,
1,439,85.51,65.00
2,1483,131.14,108.00
3,428,180.44,169.00
4,650,227.84,200.00
5,245,254.80,249.00
6,203,275.32,250.00
7,39,323.82,260.00
8,55,348.58,300.00
9,10,351.10,240.00


## 6. How do superhost and non-superhost listings compare?

In [8]:
df.groupby("host_is_superhost").agg(
    listings=("id", "count"),
    avg_price_usd=("price_usd", "mean"),
    median_price_usd=("price_usd", "median"),
    rated_listings=("review_scores_rating", "count"),
    avg_rating=("review_scores_rating", "mean"),
    avg_reviews=("number_of_reviews", "mean"),
    avg_availability_365=("availability_365", "mean")
)

,listings,avg_price_usd,median_price_usd,rated_listings,avg_rating,avg_reviews,avg_availability_365
host_is_superhost,,,,,,,
False,3178,173.60,150.00,2378,91.06,15.73,176.67
True,407,176.43,139.00,394,97.08,44.94,200.24


## 7. How do instant-bookable listings compare?

In [9]:
df.groupby("instant_bookable").agg(
    listings=("id", "count"),
    avg_price_usd=("price_usd", "mean"),
    median_price_usd=("price_usd", "median"),
    rated_listings=("review_scores_rating", "count"),
    avg_rating=("review_scores_rating", "mean"),
    avg_reviews=("number_of_reviews", "mean")
)

,listings,avg_price_usd,median_price_usd,rated_listings,avg_rating,avg_reviews
instant_bookable,,,,,,
False,2991,177.72,150.00,2297,92.04,16.11
True,594,154.84,125.00,475,91.32,33.81


## 8. What is the cancellation-policy mix?

In [10]:
df.groupby("cancellation_policy").agg(
    listings=("id", "count"),
    avg_price_usd=("price_usd", "mean"),
    median_price_usd=("price_usd", "median"),
    avg_rating=("review_scores_rating", "mean")
).sort_values("listings", ascending=False)

,listings,avg_price_usd,median_price_usd,avg_rating
cancellation_policy,,,,
strict,1582,188.22,169.00,91.58
flexible,999,148.91,100.00,91.59
moderate,919,162.62,135.00,92.96
super_strict_30,85,324.14,299.00,89.22


## 9. How concentrated are listings among hosts?

In [11]:
host_summary = df.groupby("host_id").agg(
    listings=("id", "count"),
    avg_price_usd=("price_usd", "mean")
).sort_values("listings", ascending=False)
print("Unique hosts:", len(host_summary))
print("Hosts with one listing (%):", round(host_summary["listings"].eq(1).mean()*100, 2))
host_summary.head(10)

Unique hosts: 2181
Hosts with one listing (%): 81.06


,listings,avg_price_usd
host_id,,
30283594,136,236.79
25188,79,303.05
9419684,61,318.16
12243051,58,218.12
22348222,50,152.50
4962900,25,156.48
21184200,24,227.75
1444340,24,201.67
26956083,24,103.04


## 10. How complete are the review metrics?

In [12]:
review_fields = [c for c in df.columns if c.startswith("review_scores_")]
pd.DataFrame({
    "available": df[review_fields].notna().sum(),
    "missing": df[review_fields].isna().sum(),
    "coverage_pct": df[review_fields].notna().mean()*100
}).sort_values("coverage_pct", ascending=False)

,available,missing,coverage_pct
review_scores_rating,2772,813,77.32
review_scores_cleanliness,2767,818,77.18
review_scores_communication,2767,818,77.18
review_scores_checkin,2765,820,77.13
review_scores_value,2764,821,77.10
review_scores_location,2763,822,77.07
review_scores_accuracy,2762,823,77.04


## Interpretation guardrails

- Results refer to the 2016 snapshot only.
- Missing review values are excluded from rating means.
- Availability is not treated as occupancy.
- Group differences are descriptive and should not be described as causal effects.
- Extreme nightly prices remain in the source-of-truth dataset.